### Como essa função se integra ao tracker

Para cada frame, é feita uma busca circular a partir da posição do objeto no frame anterior (prev_bbox).
Essa função realiza a busca de forma eficiente: em vez de gerar uma lista com todas as regiões — o que demoraria muito tempo e nem sempre seria necessário — ela gera uma região por vez usando generate_search_regions_circular.

Cada região é então classificada. Caso a região seja satisfatória (atingindo um score adequado), a busca é interrompida e essa região é considerada a posição do objeto. Caso contrário, gera-se a próxima região até encontrar uma adequada ou atingir o raio máximo de busca.

In [ ]:
import numpy as np  # importa o NumPy para operações numéricas e trigonométricas

def generate_search_regions_circular(
    prev_bbox, 
    frame_shape, 
    search_region_scale=1.5, 
    step_size=1,
    start_angle=0,
    clockwise=True
):
    """
    Gera regiões de busca circulares em torno do bbox anterior,
    onde step_size define o deslocamento em pixels reais
    (1 px = step_size=1).
    """

    x, y, w, h = prev_bbox
    center_x, center_y = x + w // 2, y + h // 2

    raio_max = (max(w, h) / 2) * search_region_scale

    yield (x, y, w, h)  # primeiro retorna o bbox original

    # passo radial em pixels
    for raio in np.arange(step_size, raio_max + step_size, step_size):
        circunferencia = 2 * np.pi * raio

        # define espaçamento angular de forma que os pontos na borda
        # fiquem separados por ~step_size pixels ao longo da circunferência
        num_steps = max(8, int(np.ceil(circunferencia / step_size)))
        print(num_steps)
        direction = -1 if clockwise else 1

        thetas = start_angle + direction * np.linspace(0, 2 * np.pi, num_steps, endpoint=False)
        print(thetas)

        # deslocamentos em pixels
        pxs = (center_x + raio * np.cos(thetas) - w / 2).astype(int)
        pys = (center_y + raio * np.sin(thetas) - h / 2).astype(int)

        # garante que o bbox não ultrapasse os limites do frame
        pxs = np.clip(pxs, 0, frame_shape[1] - w)
        pys = np.clip(pys, 0, frame_shape[0] - h)

        for px, py in zip(pxs, pys):
            yield (px, py, w, h)


In [ ]:
import cv2
img_path = "C:/Users/Isabella/tcc/wisard_object_tracker/data/tiger2/imgs/img00000.png"
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Imagem não encontrada: {img_path}")

bbox = (16, 30, 34, 39)
x, y, w, h = bbox

generate_search_regions_circular(prev_bbox=bbox,  frame_shape=img.shape)

In [ ]:
# --- Caminho e imagem base ---
import cv2
img_path = "C:/Users/Isabella/tcc/wisard_object_tracker/data/tiger2/imgs/img00000.png"
img = cv2.imread(img_path)
if img is None:
    raise FileNotFoundError(f"Imagem não encontrada: {img_path}")

bbox = (16, 30, 34, 39)
x, y, w, h = bbox

height, width = img.shape[:2]
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
fps = 1

# --- Conjunto de parâmetros para testar ---
search_scales = [2, 5, 10]
steps = [3, 10, 20]
angles = [0, np.pi/2, np.pi]  # direita, baixo, esquerda
directions = [True, False]    # horário / anti-horário
import os
# --- Pasta de saída ---
os.makedirs("outputs_search", exist_ok=True)

# --- Loop principal ---
for scale in search_scales:
    for step in steps:
        for start_angle in angles:
            for clockwise in directions:
                # Nome e título
                direction_str = "clockwise" if clockwise else "anticlockwise"
                angle_deg = int(np.degrees(start_angle))
                video_name = f"search_scale{scale}_step{step}_angle{angle_deg}_{direction_str}.mp4"
                output_path = os.path.join("outputs_search", video_name)

                # Inicia vídeo
                out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

                # Gera regiões
                regions = generate_search_regions_circular(
                    bbox, frame_shape,
                    search_region_scale=scale,
                    step_size=step,
                    start_angle=start_angle,
                    clockwise=clockwise
                )

                for i, region in enumerate(regions):
                    frame = img.copy()
                    cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 0), 2)  # original azul
                    (px, py, pw, ph) = region
                    cv2.rectangle(frame, (px, py), (px + pw, py + ph), (0, 0, 255), 1)  # atual vermelho

                    # Escreve parâmetros no frame
                    # Escreve os parâmetros no canto superior esquerdo
                    param_text = f"scale={scale}, step={step}, start_angle={start_angle}, clockwise={clockwise}"
                    text_position = (10, frame.shape[0] - 10)  # 10 px acima da borda inferior
                    cv2.putText(frame, param_text, text_position,
                                cv2.FONT_HERSHEY_SIMPLEX, 0.3, (0, 255, 0), 1, cv2.LINE_AA)


                    out.write(frame)

                out.release()
                print(f"✅ Vídeo salvo: {output_path}")

print("🎬 Todos os vídeos foram gerados em 'outputs_search_videos'")